# Data Leakage and Normalization in Deep Learning

---

## Learning Objectives

By the end of this notebook, you will:

- Understand what **data leakage** is and why it is dangerous
- Identify common sources of leakage in ML/DL pipelines
- Implement the **correct pipeline**: split first, then fit transformations on train only
- Use `StandardScaler` and `MinMaxScaler` properly
- Distinguish between **input normalization** and **batch normalization** (they serve different purposes)
- Perform sanity checks: baseline models, random labels test

## Prerequisites

- **DL100**: Neural network fundamentals
- **DL150**: PyTorch foundations
- **Notebook 01-02**: MLP for regression and classification

## Table of Contents

1. [Setup and Imports](#1.-Setup-and-Imports)
2. [What is Data Leakage?](#2.-What-is-Data-Leakage?)
3. [Common Leakage Sources](#3.-Common-Leakage-Sources)
4. [The Correct Pipeline: Split First](#4.-The-Correct-Pipeline:-Split-First)
5. [StandardScaler and MinMaxScaler](#5.-StandardScaler-and-MinMaxScaler)
6. [Wrong Way vs Right Way: Code Demo](#6.-Wrong-Way-vs-Right-Way:-Code-Demo)
7. [Batch Norm vs Input Normalization](#7.-Batch-Norm-vs-Input-Normalization)
8. [Sanity Checks](#8.-Sanity-Checks)
9. [Common Mistakes and Debugging Tips](#9.-Common-Mistakes-and-Debugging-Tips)
10. [Exercises](#10.-Exercises)

---

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.datasets import make_classification

set_global_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
print("Setup complete.")

---

## 2. What is Data Leakage?

**Data leakage** occurs when information from outside the training set is used to create the model. This leads to:

- **Overly optimistic** performance estimates during development
- **Poor performance** in production (on truly unseen data)
- **Wasted time and resources** building models that do not actually work

### The Core Principle

$$\text{Anything computed from or influenced by the test/validation data must NOT be used during training.}$$

Think of it this way: the test set represents the **future**. You cannot use future information to make decisions about the present.

### Why It Is Especially Dangerous

- Leakage can be **subtle** -- your code runs fine, metrics look great
- It often only manifests when the model hits **production data**
- The gap between dev metrics and production metrics can be enormous

---

## 3. Common Leakage Sources

| Source | What Happens | How to Fix |
|--------|-------------|------------|
| Normalizing before split | Scaler sees test data statistics | Fit scaler on train only, then transform val/test |
| Feature engineering on full data | Derived features contain test info | Engineer features after splitting |
| Using test data for model selection | Test set effectively becomes validation | Use a separate validation set |
| Time series: random split | Future data leaks into training | Use time-based (chronological) splits |
| Duplicate samples across splits | Same sample in train and test | Deduplicate before splitting |
| Data augmentation before split | Augmented versions of test images in train | Augment only after splitting |

In [ ]:
# Create a synthetic dataset for demonstration
set_global_seed(42)

X, y = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_classes=2,
    random_state=42,
    flip_y=0.1,  # 10% label noise to make it more realistic
)

# Make features have different scales (common in real data)
scales = np.random.RandomState(42).uniform(0.1, 100, size=20)
X = X * scales

print(f"Dataset shape: {X.shape}")
print(f"Feature scales (min, max): ({X.min():.1f}, {X.max():.1f})")
print(f"Feature std per column (first 5): {X.std(axis=0)[:5].round(2)}")

---

## 4. The Correct Pipeline: Split First

### The Wrong Way (Leakage)

```
1. Normalize ALL data        <-- WRONG: scaler sees test statistics
2. Split into train/val/test
3. Train model
```

### The Right Way (No Leakage)

```
1. Split into train/val/test   <-- FIRST
2. Fit scaler on TRAIN only
3. Transform train, val, test using the train-fitted scaler
4. Train model
```

Mathematically, for StandardScaler:

$$\mu_{\text{train}} = \frac{1}{N_{\text{train}}} \sum_{i=1}^{N_{\text{train}}} x_i, \quad \sigma_{\text{train}} = \sqrt{\frac{1}{N_{\text{train}}} \sum_{i=1}^{N_{\text{train}}} (x_i - \mu_{\text{train}})^2}$$

$$x_{\text{scaled}} = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}} \quad \text{(applied to ALL splits using train stats)}$$

---

## 5. StandardScaler and MinMaxScaler

### StandardScaler (Z-score normalization)

$$x' = \frac{x - \mu}{\sigma}$$

- Centers data around 0, scales to unit variance
- Good default choice for most neural networks
- Robust to different feature scales

### MinMaxScaler

$$x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

- Scales to $[0, 1]$ range
- Preserves zero entries in sparse data
- Sensitive to outliers

In [ ]:
# Demonstrate both scalers
example_data = np.array([[1, 200, 0.5],
                          [2, 400, 0.8],
                          [3, 100, 0.2],
                          [4, 300, 0.6]])

print("Original data:")
print(example_data)

# StandardScaler
ss = StandardScaler()
scaled_ss = ss.fit_transform(example_data)
print(f"\nStandardScaler (mean={ss.mean_.round(2)}, std={ss.scale_.round(2)}):")
print(scaled_ss.round(3))

# MinMaxScaler
mms = MinMaxScaler()
scaled_mms = mms.fit_transform(example_data)
print(f"\nMinMaxScaler (min={mms.data_min_}, max={mms.data_max_}):")
print(scaled_mms.round(3))

---

## 6. Wrong Way vs Right Way: Code Demo

We train the same MLP on the same data, but compare:

1. **Wrong**: normalize all data before splitting (leakage)
2. **Right**: split first, fit scaler on train only

In [ ]:
class SimpleMLP(nn.Module):
    """Simple MLP for binary classification."""

    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 2),
        )

    def forward(self, x):
        return self.network(x)


def train_and_evaluate(X_train, y_train, X_val, y_val, X_test, y_test,
                       epochs=30, lr=1e-3, seed=42):
    """Train MLP and return val/test accuracy."""
    set_global_seed(seed)

    train_ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long),
    )
    val_ds = TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.long),
    )
    test_ds = TensorDataset(
        torch.tensor(X_test, dtype=torch.float32),
        torch.tensor(y_test, dtype=torch.long),
    )

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=64)
    test_loader = DataLoader(test_ds, batch_size=64)

    model = SimpleMLP(X_train.shape[1]).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Train
    for epoch in range(epochs):
        model.train()
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(X_b), y_b)
            loss.backward()
            optimizer.step()

    # Evaluate
    model.eval()
    results = {}
    for name, loader in [("val", val_loader), ("test", test_loader)]:
        correct, total = 0, 0
        with torch.no_grad():
            for X_b, y_b in loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                preds = model(X_b).argmax(-1)
                correct += (preds == y_b).sum().item()
                total += y_b.size(0)
        results[f"{name}_acc"] = correct / total

    return results

In [ ]:
# ============================================================
# WRONG WAY: Normalize BEFORE splitting (DATA LEAKAGE!)
# ============================================================
print("=" * 60)
print("WRONG WAY: Normalize before splitting (LEAKAGE!)")
print("=" * 60)

scaler_wrong = StandardScaler()
X_scaled_wrong = scaler_wrong.fit_transform(X)  # fit on ALL data!

X_train_w, X_temp_w, y_train_w, y_temp_w = train_test_split(
    X_scaled_wrong, y, test_size=0.3, random_state=42, stratify=y
)
X_val_w, X_test_w, y_val_w, y_test_w = train_test_split(
    X_temp_w, y_temp_w, test_size=0.5, random_state=42, stratify=y_temp_w
)

print(f"Scaler was fit on ALL {X.shape[0]} samples (including val+test) -- LEAKAGE!")

results_wrong = train_and_evaluate(
    X_train_w, y_train_w, X_val_w, y_val_w, X_test_w, y_test_w
)
print(f"Val Accuracy:  {results_wrong['val_acc']:.4f}")
print(f"Test Accuracy: {results_wrong['test_acc']:.4f}")

In [ ]:
# ============================================================
# RIGHT WAY: Split FIRST, then normalize using train stats only
# ============================================================
print("=" * 60)
print("RIGHT WAY: Split first, normalize with train stats only")
print("=" * 60)

# Step 1: Split FIRST
X_train_r, X_temp_r, y_train_r, y_temp_r = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val_r, X_test_r, y_val_r, y_test_r = train_test_split(
    X_temp_r, y_temp_r, test_size=0.5, random_state=42, stratify=y_temp_r
)

# Step 2: Fit scaler on TRAIN only
scaler_right = StandardScaler()
X_train_r = scaler_right.fit_transform(X_train_r)  # fit AND transform train
X_val_r = scaler_right.transform(X_val_r)           # transform only (no fit!)
X_test_r = scaler_right.transform(X_test_r)         # transform only (no fit!)

print(f"Scaler was fit on TRAIN only ({X_train_r.shape[0]} samples) -- CORRECT!")

results_right = train_and_evaluate(
    X_train_r, y_train_r, X_val_r, y_val_r, X_test_r, y_test_r
)
print(f"Val Accuracy:  {results_right['val_acc']:.4f}")
print(f"Test Accuracy: {results_right['test_acc']:.4f}")

In [ ]:
# Compare results
print("\n" + "=" * 60)
print("COMPARISON")
print("=" * 60)
print(f"{'Method':<30} {'Val Acc':<12} {'Test Acc':<12}")
print("-" * 54)
print(f"{'Wrong (leakage)':<30} {results_wrong['val_acc']:<12.4f} {results_wrong['test_acc']:<12.4f}")
print(f"{'Right (no leakage)':<30} {results_right['val_acc']:<12.4f} {results_right['test_acc']:<12.4f}")

print("\nNote: On this synthetic dataset the difference may be small.")
print("In real-world scenarios with smaller datasets or time-series data,")
print("leakage can cause huge overestimates of model performance.")

---

## 7. Batch Norm vs Input Normalization

These are **different things** that serve different purposes:

| Aspect | Input Normalization | Batch Normalization |
|--------|-------------------|--------------------|
| **What** | Preprocesses raw input features | Normalizes hidden layer activations |
| **When** | Before training (data preprocessing) | During training (inside the network) |
| **Statistics** | Computed from training set, fixed | Computed per mini-batch during training, running stats for inference |
| **Purpose** | Makes features comparable in scale | Stabilizes training, enables higher LRs |
| **Implemented by** | `sklearn.preprocessing.StandardScaler` | `torch.nn.BatchNorm1d` |

### Batch Normalization formula

For a mini-batch $\mathcal{B} = \{x_1, \ldots, x_m\}$:

$$\hat{x}_i = \frac{x_i - \mu_{\mathcal{B}}}{\sqrt{\sigma_{\mathcal{B}}^2 + \epsilon}} \cdot \gamma + \beta$$

where $\gamma$ (scale) and $\beta$ (shift) are **learnable parameters**.

In [ ]:
# Demonstrate the difference
print("Input Normalization (StandardScaler):")
print("  - Applied ONCE before training")
print("  - Uses train set statistics")
print("  - External to the model")

print("\nBatch Normalization (nn.BatchNorm1d):")
bn = nn.BatchNorm1d(10)
print(f"  - Learnable parameters: gamma={bn.weight.shape}, beta={bn.bias.shape}")
print(f"  - Running mean: {bn.running_mean.shape}")
print(f"  - Running var: {bn.running_var.shape}")
print(f"  - Epsilon: {bn.eps}")
print("  - Applied at EVERY forward pass inside the network")
print("  - Uses mini-batch stats (train) or running stats (eval)")

# Important: you should use BOTH
print("\nBest practice: Use input normalization AND batch norm together.")
print("They complement each other -- they are NOT substitutes.")

In [ ]:
# Compare: no normalization vs input norm vs input norm + batch norm

class MLPWithBatchNorm(nn.Module):
    """MLP with batch normalization."""

    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 2),
        )

    def forward(self, x):
        return self.network(x)


# Prepare three scenarios
# 1. No normalization at all
X_train_raw_split, X_temp_raw, y_train_raw_split, y_temp_raw = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val_raw_split, X_test_raw_split, y_val_raw_split, y_test_raw_split = train_test_split(
    X_temp_raw, y_temp_raw, test_size=0.5, random_state=42, stratify=y_temp_raw
)

print("Training three configurations...\n")

# Config 1: No normalization
print("1. No normalization:")
r1 = train_and_evaluate(X_train_raw_split, y_train_raw_split,
                        X_val_raw_split, y_val_raw_split,
                        X_test_raw_split, y_test_raw_split)
print(f"   Val: {r1['val_acc']:.4f}, Test: {r1['test_acc']:.4f}")

# Config 2: Input normalization only
print("2. Input normalization (StandardScaler):")
r2 = train_and_evaluate(X_train_r, y_train_r, X_val_r, y_val_r, X_test_r, y_test_r)
print(f"   Val: {r2['val_acc']:.4f}, Test: {r2['test_acc']:.4f}")

# Config 3: Input normalization + batch norm (need custom training)
print("3. Input normalization + BatchNorm:")
set_global_seed(42)
model_bn = MLPWithBatchNorm(X_train_r.shape[1]).to(device)
train_ds_bn = TensorDataset(
    torch.tensor(X_train_r, dtype=torch.float32),
    torch.tensor(y_train_r, dtype=torch.long),
)
val_ds_bn = TensorDataset(
    torch.tensor(X_val_r, dtype=torch.float32),
    torch.tensor(y_val_r, dtype=torch.long),
)
test_ds_bn = TensorDataset(
    torch.tensor(X_test_r, dtype=torch.float32),
    torch.tensor(y_test_r, dtype=torch.long),
)
train_loader_bn = DataLoader(train_ds_bn, batch_size=64, shuffle=True)
val_loader_bn = DataLoader(val_ds_bn, batch_size=64)
test_loader_bn = DataLoader(test_ds_bn, batch_size=64)

opt_bn = torch.optim.Adam(model_bn.parameters(), lr=1e-3)
loss_fn_bn = nn.CrossEntropyLoss()

for epoch in range(30):
    model_bn.train()
    for X_b, y_b in train_loader_bn:
        X_b, y_b = X_b.to(device), y_b.to(device)
        opt_bn.zero_grad()
        loss_fn_bn(model_bn(X_b), y_b).backward()
        opt_bn.step()

model_bn.eval()
r3 = {}
for name, loader in [("val", val_loader_bn), ("test", test_loader_bn)]:
    correct, total = 0, 0
    with torch.no_grad():
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            correct += (model_bn(X_b).argmax(-1) == y_b).sum().item()
            total += y_b.size(0)
    r3[f"{name}_acc"] = correct / total
print(f"   Val: {r3['val_acc']:.4f}, Test: {r3['test_acc']:.4f}")

---

## 8. Sanity Checks

Before trusting your model, run these sanity checks:

### Check 1: Baseline Model

A model that always predicts the majority class should give you a lower bound on accuracy.

### Check 2: Random Labels Test

If your model achieves high accuracy on **randomly shuffled labels**, something is wrong (likely leakage or a bug).

In [ ]:
# Sanity Check 1: Baseline (majority class predictor)
print("SANITY CHECK 1: Baseline Model")
print("=" * 40)

unique, counts = np.unique(y_train_r, return_counts=True)
majority_class = unique[np.argmax(counts)]
majority_fraction = counts.max() / counts.sum()

print(f"Majority class: {majority_class}")
print(f"Baseline accuracy (always predict majority): {majority_fraction:.4f}")
print(f"Our model test accuracy: {results_right['test_acc']:.4f}")

if results_right['test_acc'] > majority_fraction:
    print("Model beats baseline -- good!")
else:
    print("WARNING: Model does not beat baseline!")

In [ ]:
# Sanity Check 2: Random Labels Test
print("SANITY CHECK 2: Random Labels Test")
print("=" * 40)

# Shuffle labels randomly
rng = np.random.RandomState(42)
y_train_random = rng.permutation(y_train_r)
y_val_random = rng.permutation(y_val_r)
y_test_random = rng.permutation(y_test_r)

results_random = train_and_evaluate(
    X_train_r, y_train_random, X_val_r, y_val_random, X_test_r, y_test_random,
    epochs=30
)

print(f"Random labels - Val Acc:  {results_random['val_acc']:.4f}")
print(f"Random labels - Test Acc: {results_random['test_acc']:.4f}")
print(f"Expected (random guessing for 2 classes): ~0.5000")

if results_random['test_acc'] < 0.6:
    print("Model cannot learn random labels -- good! No leakage detected.")
else:
    print("WARNING: Model achieves high acc on random labels -- possible leakage!")

---

## 9. Common Mistakes and Debugging Tips

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Calling `scaler.fit_transform()` on val/test | Leakage: scaler uses val/test stats | Use `scaler.transform()` (no fit) on val/test |
| Normalizing before splitting | Leakage: scaler sees all data | Always split first |
| Using test set for hyperparameter tuning | Test metrics are meaningless | Use a separate validation set |
| Forgetting `model.eval()` with BatchNorm | BN uses batch stats instead of running stats | Always toggle `train()`/`eval()` |
| Not saving the scaler for inference | Cannot transform new data consistently | Save scaler with `joblib.dump()` or `pickle` |
| Confusing input norm and batch norm | Thinking one replaces the other | They are complementary; use both |
| Feature engineering on full dataset | Derived features leak test info | Compute features after splitting |

---

## 10. Exercises

### Exercise 1: Identify the Leakage

Each code snippet below contains a data leakage bug. Identify the issue and explain how to fix it.

**Snippet A:**
```python
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_all, test_size=0.2)
```

**Snippet B:**
```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)  # <--- spot the bug
```

**Snippet C:**
```python
model.fit(X_train, y_train)
# Tune threshold on test set
best_threshold = find_best_threshold(model, X_test, y_test)
# Report final performance
print(f"Test accuracy: {evaluate(model, X_test, y_test, threshold=best_threshold)}")
```

In [ ]:
# ===== EXERCISE 1: Your answers here =====
# Snippet A issue: ...
# Snippet B issue: ...
# Snippet C issue: ...
pass

### Exercise 2: Compare Scalers

Train the `SimpleMLP` on the synthetic data using:

1. No scaling
2. `StandardScaler`
3. `MinMaxScaler`

Use the correct pipeline (split first). Compare val/test accuracy.

In [ ]:
# ===== EXERCISE 2: Your code here =====
# scalers = {"None": None, "StandardScaler": StandardScaler(), "MinMaxScaler": MinMaxScaler()}
# for name, scaler in scalers.items():
#     ...
pass

### Exercise 1 -- Solution

**Snippet A:** The scaler is fit on ALL data (`X_all`) before splitting. The mean and std used for normalization include information from the test set. **Fix:** Split first, then `fit_transform` on train and `transform` on test.

**Snippet B:** The scaler is `fit_transform`'d on the test set separately. This means the test set is normalized using its own statistics, not the training statistics. **Fix:** Use `scaler.transform(X_test)` (no `fit`).

**Snippet C:** The decision threshold is tuned on the test set. The reported "test accuracy" is optimistic because the threshold was chosen to maximize performance on that specific set. **Fix:** Use a validation set to tune the threshold, then evaluate on the test set.

### Exercise 2 -- Solution

In [ ]:
# Split first (correct pipeline)
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_vl, X_te, y_vl, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp)

scalers = {
    "No scaling": None,
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
}

print(f"{'Scaler':<20} {'Val Acc':<12} {'Test Acc':<12}")
print("-" * 44)

for name, scaler in scalers.items():
    if scaler is not None:
        X_tr_s = scaler.fit_transform(X_tr)  # fit on train
        X_vl_s = scaler.transform(X_vl)      # transform only
        X_te_s = scaler.transform(X_te)      # transform only
    else:
        X_tr_s, X_vl_s, X_te_s = X_tr.copy(), X_vl.copy(), X_te.copy()

    r = train_and_evaluate(X_tr_s, y_tr, X_vl_s, y_vl, X_te_s, y_te)
    print(f"{name:<20} {r['val_acc']:<12.4f} {r['test_acc']:<12.4f}")

print("\nBoth StandardScaler and MinMaxScaler should outperform no scaling.")
print("StandardScaler is generally the better default for neural networks.")

---

**Next module:** [DL300 -- Computer Vision with CNNs](../DL300_Computer_Vision_CNN/01_CNN_Basics_Convolutions_Pooling.ipynb)